# Sentiment Analysis: Direct Model and Pipeline
This notebook runs the same sentence using two approaches with `cardiffnlp/twitter-roberta-base-sentiment-latest`.

The model assigns scores to **negative, neutral, and positive** sentiment. Run the cells in Google Colab. Each code cell includes its own package installation and can run independently. No Hugging Face API key is needed.

The installation uses **Transformers** to load the model, **PyTorch** to run it, **NumPy** to rank scores, and **SciPy** to apply softmax. This notebook uses PyTorch; TensorFlow is not required.

## 1. Direct model approach
This example shows the individual steps of sentiment prediction:

1. **Preprocess the text:** replace usernames with `@user` and links with `http`.
2. **Load the tokenizer:** `AutoTokenizer` converts the sentence into token IDs the model understands.
3. **Load the configuration:** `AutoConfig` provides `id2label`, which maps each output position to a sentiment label.
4. **Load the model:** `AutoModelForSequenceClassification` loads the pretrained sentiment classifier. `model.eval()` disables training behavior such as dropout.
5. **Tokenize the sentence:** `return_tensors='pt'` produces PyTorch tensors.
6. **Run inference:** `model(**encoded_input)` returns raw class scores called **logits**.
7. **Apply softmax:** convert logits into scores between 0 and 1 that sum to approximately 1.
8. **Rank and print:** `np.argsort(scores)[::-1]` orders the labels from highest score to lowest.

**Reading the output:** the first printed label is the predicted sentiment. A score such as `0.85` means the model assigned that class a confidence score of 85%; it does **not** mean the model has 85% accuracy. Accuracy requires comparing predictions with correct labels across a dataset.

In [ ]:
%pip install -q "transformers>=4.45,<5" "torch>=2.6" numpy scipy

from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer, AutoConfig
import numpy as np
from scipy.special import softmax

# Preprocess text (username and link placeholders)
def preprocess(text):
    new_text = []
    for t in text.split(" "):
        t = '@user' if t.startswith('@') and len(t) > 1 else t
        t = 'http' if t.startswith('http') else t
        new_text.append(t)
    return " ".join(new_text)

MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
config = AutoConfig.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)
model.eval()

text = "Covid cases are increasing fast!"
text = preprocess(text)
encoded_input = tokenizer(text, return_tensors='pt')
output = model(**encoded_input)
scores = output[0][0].detach().numpy()
scores = softmax(scores)

# Print labels and scores
ranking = np.argsort(scores)[::-1]
for i in range(scores.shape[0]):
    l = config.id2label[int(ranking[i])]
    s = scores[ranking[i]]
    print(f"{i+1}) {l} {np.round(float(s), 4)}")

## 2. Pipeline approach
`pipeline("sentiment-analysis", ...)` combines **tokenization, model inference, and softmax** into one interface.

- The same preprocessing is applied to the same input sentence.
- `model=MODEL` and `tokenizer=MODEL` load the same classifier and tokenizer as the direct example.
- `device=-1` runs the model on CPU.
- `top_k=None` returns scores for **all three sentiment labels**.
- The results are sorted by score and printed in the same format as the direct example.

Both approaches should produce matching scores, apart from small numerical differences. The direct approach helps explain the internal steps; pipeline makes application code shorter.

You can change `text` in either example to test another sentence. This is **inference using a pretrained model**, not training or fine-tuning. The model was developed for tweet sentiment, so banking or other domain-specific usage should be evaluated with labeled examples.

[Official model card](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest)

In [ ]:
%pip install -q "transformers>=4.45,<5" "torch>=2.6" numpy scipy

from transformers import pipeline
import numpy as np

# Preprocess text (username and link placeholders)
def preprocess(text):
    new_text = []
    for t in text.split(" "):
        t = '@user' if t.startswith('@') and len(t) > 1 else t
        t = 'http' if t.startswith('http') else t
        new_text.append(t)
    return " ".join(new_text)

MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
sentiment = pipeline("sentiment-analysis", model=MODEL, tokenizer=MODEL, device=-1)

text = "Covid cases are increasing fast!"
text = preprocess(text)
scores = sentiment(text, top_k=None)
scores = sorted(scores, key=lambda result: result['score'], reverse=True)

# Print labels and scores
for i, result in enumerate(scores):
    l = result['label']
    s = result['score']
    print(f"{i+1}) {l} {np.round(float(s), 4)}")